# ML2 - Supervised Learning

# 1. Theory

## 1) linear regression - analytical solution

$\large X$ - features\
$\large Y$ - target\
$\large θ$ - weights vector
1. **MSE Loss***: $$\large L(θ, X, Y)\ =\ ∥Xθ−Y∥^2_2$$ \
use matrix calculus to achive
2. **Loss Gradient**: $$\large ∇L=2X^T(Xθ−Y)$$
3. **Set to zero**: $$\large 2X^T(Xθ−Y)=0$$
4. **Rearrange**: $$\large X^TXθ=X^TY$$
5. **Solve**: $$\large θ=(X^TX)^{−1}X^TY$$\
*for analytical solution there is no need for taking the mean of squared errors - loss critical point will be the same

## 2) what changes for ridge/lasso regularization?

1. **MSE Ridge Loss**: $$\large L({θ}, X, Y) = || (X{θ} - Y) ||^2_2 + {λ}||{θ}||^2_2$$
2. **Loss Gradient**: $$\large ∇L = 2X^T(X{θ} - Y) + 2{λ}{θ}$$
3. **Set to zero**: $$\large X^TX{θ} - X^TY + {λ}{θ}=0$$
4. **Rearrange**: $$\large {θ}(X^TX + {λ}) = X^TY$$
5. **Solve**: $$\large {θ} = (X^TX + {λ}I)^{-1}X^TY$$

- basically, since taking derivative is linear in addition, we just add the derivative ot L2 norm squared of weights vector to L gradient from basic linear regression

- in **Lasso case**, since **L1 norm is not a smooth function**, there are points where L1 norm and resulting Lasso loss function are not differentiable -> no pure form analytical solution
- **L1 penalty**
  $$\Large {λ}||{θ}||^1_1 = \sum_{i=1}^{d}{|θ_i|}$$
- adding L1 penaly leads to some coefficients being zeroed, thus L1 regularization is applicable to **feature selection**

## 3) nonlinear dependencies with linear models

If we suspect that the target variable $y$ is not expressed as a linear function of $x_1, x_2$, but also depends on $log(x_1x_2)$ and possibly on whether the features have different signs, then we can introduce additional terms into our linear relationship. We simply declare these terms as new variables and add corresponding regression coefficients before them:

$\large y ≈ w_1x_1 + w_2x_2 + w_3\log(x_1) + w_4\operatorname{sgn}(x_1x_2)$,

and thus, from a two-dimensional nonlinear problem, we obtain a four-dimensional regression problem.

# 2. Import libraries, load data

In [2]:
import numpy as np
import pandas as pd
import sklearn
from collections import Counter

In [3]:
import warnings
warnings.filterwarnings('ignore')

In [4]:
pd.set_option('display.float_format', '{:.3f}'.format)

In [5]:
train = pd.read_json('../datasets/train.json')
test = pd.read_json('../datasets/test.json')

# 3. Create new features

In [6]:
train.head(1)

,bathrooms,bedrooms,building_id,created,description,display_address,features,latitude,listing_id,longitude,manager_id,photos,price,street_address,interest_level
4,1.000,1,8579a0b0d54db803821a35a4a615e97a,2016-06-16 05:55:27,Spacious 1 Bedroom 1 Bathroom in Williamsburg!...,145 Borinquen Place,"[Dining Room, Pre-War, Laundry in Building, Di...",40.711,7170325,-73.954,a10db4590843d78c784171a107bdacb4,[https://photos.renthop.com/2/7170325_3bb5ac84...,2400,145 Borinquen Place,medium


In [7]:
all_features = []
for idx, row in train.iterrows():
    all_features += row['features']

In [8]:
top_20 = [feature for feature, count in Counter(all_features).most_common(20)]

In [9]:
for feature in top_20:
    train[feature.replace(' ', '')] = train.apply(lambda row: int((feature in row['features'])), axis=1)
    test[feature.replace(' ', '')] = test.apply(lambda row: int((feature in row['features'])), axis=1)

In [10]:
top_20 = [f.replace(' ', '') for f in top_20]
feature_list = ['bathrooms', 'bedrooms'] + top_20

In [11]:
top_20

['Elevator',
 'CatsAllowed',
 'HardwoodFloors',
 'DogsAllowed',
 'Doorman',
 'Dishwasher',
 'NoFee',
 'LaundryinBuilding',
 'FitnessCenter',
 'Pre-War',
 'LaundryinUnit',
 'RoofDeck',
 'OutdoorSpace',
 'DiningRoom',
 'HighSpeedInternet',
 'Balcony',
 'SwimmingPool',
 'LaundryInBuilding',
 'NewConstruction',
 'Terrace']

#### - remove outliers from `train`

In [100]:
train = train[train['price'].between(train['price'].quantile(0.01), train['price'].quantile(0.99))]
train = train[train['bathrooms'].between(train['bathrooms'].quantile(0.01), train['bathrooms'].quantile(0.99))]
train = train[train['bedrooms'].between(train['bedrooms'].quantile(0.01), train['bedrooms'].quantile(0.99))]

In [101]:
X_train = train[feature_list]
X_test = test[feature_list]
y_train = train['price']
y_test = test['price']

# 4. Linear Regression

In [102]:
from my_regression import MyLinearRegressor, r_squared

## 1) $R^2$ - Coefficient of Determination

- $R^2$ is a statistical measure that indicates the **proportion of the variance** in a target that is explained by the features in a model
- range is $[-\infty; 1]$, where $1$ indicates a perfect fit; $0$ is a baseline value indicating that the model explains none of the variability (like a naive mean predictor)
- is used to assess the **goodness of fit** for a model

$$\Large R^2 = 1 - \frac{SSE}{SST} = 1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2}$$

$\large SSE$ - *squared sum of errors*

$\large SST$ - *total variation* in the data

$\large y_i$ - $y$ value for observation $i$

$\large \hat{y}_i$ - predicted $y$ value for observation $i$

$\large \bar{y}$ - mean of $y$

## 2) my Linear Regression implementation 

### - initialize and fit MyLinearRegressor with two methods - analytical and SGD

In [103]:
my_lin_reg_analytical = MyLinearRegressor(method='analytical')

In [104]:
my_lin_reg_analytical.fit(X_train, y_train);

In [105]:
my_lin_reg_SGD = MyLinearRegressor(method='SGD', eta=0.001, tol=200, log_freq=5)

In [106]:
my_lin_reg_SGD.fit(X_train, y_train);

Epoch 0, Loss: 1552899.2915
Epoch 5, Loss: 1010789.6369
Epoch 10, Loss: 1005567.6284
Epoch 15, Loss: 1004798.8091


- SGD took only 19 epochs to converge with given hyperparams

In [107]:
my_lin_reg_SGD.n_iter_

19

### - initialize and fit sklearn.linear_model.LinearRegression

In [108]:
lin_reg = sklearn.linear_model.LinearRegression()

In [109]:
lin_reg.fit(X_train, y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


### - calculate metrics and fill in the table

In [110]:
results_r_squared = pd.DataFrame({'model': [], 'train': [], 'test': [], 'test_processed': []})
results_RMSE = pd.DataFrame({'model': [], 'train': [], 'test': [], 'test_processed': []})
results_MAE = pd.DataFrame({'model': [], 'train': [], 'test': [], 'test_processed': []})

***значения для test как в чеклисте можно получить только если убрать выбросы из test - но это неправильный подход!***

In [111]:
test_processed = test[test['bathrooms'].between(test['bathrooms'].quantile(0.01), test['bathrooms'].quantile(0.99))]
test_processed = test_processed[test_processed['bedrooms'].between(test_processed['bedrooms'].quantile(0.01), test_processed['bedrooms'].quantile(0.99))]
test_processed = test_processed[test_processed['price'].between(test_processed['price'].quantile(0.01), test_processed['price'].quantile(0.99))]
X_test_processed = test_processed[feature_list]
y_test_processed = test_processed['price']

In [112]:
for table, metric in zip((results_r_squared, results_RMSE, results_MAE), (r_squared, sklearn.metrics.root_mean_squared_error, sklearn.metrics.mean_absolute_error)):
    for model, name, idx in zip((lin_reg, my_lin_reg_analytical, my_lin_reg_SGD), ('sklearn_default_lin_reg', 'my_lin_reg_analytical', 'my_lin_reg_SGD'), (0, 1, 2)):
        table.loc[idx] = [
            name,
            metric(y_pred=model.predict(X_train), y_true=y_train),
            metric(y_pred=model.predict(X_test), y_true=y_test),
            metric(y_pred=model.predict(X_test_processed), y_true=y_test_processed)
        ]

In [113]:
results_r_squared

,model,train,test,test_processed
0,sklearn_default_lin_reg,0.575,0.022,0.571
1,my_lin_reg_analytical,0.575,0.022,0.571
2,my_lin_reg_SGD,0.575,0.022,0.572


In [114]:
results_MAE

,model,train,test,test_processed
0,sklearn_default_lin_reg,693.510,907.138,680.276
1,my_lin_reg_analytical,693.510,907.138,680.276
2,my_lin_reg_SGD,690.221,904.594,676.841


In [115]:
results_RMSE

,model,train,test,test_processed
0,sklearn_default_lin_reg,1002.852,9606.256,948.598
1,my_lin_reg_analytical,1002.852,9606.256,948.598
2,my_lin_reg_SGD,1003.378,9606.887,947.880


# 5. Ridge, Lasso, ElasticNet

In [116]:
from my_regression import MyRidge

## 1) my Ridge Regression implementation 

### - initialize and fit MyRidge with two methods - analytical and SGD

In [117]:
my_ridge_analytical = MyRidge(method='analytical')

In [118]:
my_ridge_analytical.fit(X_train, y_train);

In [119]:
my_ridge_SGD = MyRidge(method='SGD', alpha=0.01, eta=0.001, batch_size=128, tol=300, log_freq=10)

In [120]:
my_ridge_SGD.fit(X_train, y_train);

Epoch 0, Loss: 2576753.7145
Epoch 10, Loss: 1039823.0433
Epoch 20, Loss: 1015637.3631
Epoch 30, Loss: 1010601.3484


### - initialize and fit sklearn.linear_model.Ridge

In [121]:
ridge = sklearn.linear_model.Ridge()

In [122]:
ridge.fit(X_train, y_train)

,alpha,1.0
,fit_intercept,True
,copy_X,True
,max_iter,None
,tol,0.0001
,solver,'auto'
,positive,False
,random_state,None


### - calculate metrics and fill in the table

In [123]:
for table, metric in zip((results_r_squared, results_RMSE, results_MAE), (r_squared, sklearn.metrics.root_mean_squared_error, sklearn.metrics.mean_absolute_error)):
    for model, name, idx in zip((ridge, my_ridge_analytical, my_ridge_SGD), ('sklearn_default_ridge', 'my_ridge_analytical', 'my_ridge_SGD'), (3, 4, 5)):
        table.loc[idx] = [
            name,
            metric(y_pred=model.predict(X_train), y_true=y_train),
            metric(y_pred=model.predict(X_test), y_true=y_test),
            metric(y_pred=model.predict(X_test_processed), y_true=y_test_processed)
        ]

In [124]:
results_r_squared[3:6]

,model,train,test,test_processed
3,sklearn_default_ridge,0.575,0.022,0.571
4,my_ridge_analytical,0.575,0.022,0.571
5,my_ridge_SGD,0.574,0.022,0.572


In [125]:
results_MAE[3:6]

,model,train,test,test_processed
3,sklearn_default_ridge,693.507,907.136,680.270
4,my_ridge_analytical,693.506,907.134,680.270
5,my_ridge_SGD,693.039,907.594,679.125


In [126]:
results_RMSE[3:6]

,model,train,test,test_processed
3,sklearn_default_ridge,1002.852,9606.253,948.591
4,my_ridge_analytical,1002.852,9606.255,948.593
5,my_ridge_SGD,1004.626,9605.439,947.521


## 2) my Lasso Regression implementation 

In [127]:
from my_regression import MyLasso

### - initialize and fit MyLasso with SGD

In [128]:
my_lasso_SGD = MyLasso(method='SGD', alpha=0.1, eta=0.0001, batch_size=128, tol=100, log_freq=20)

In [129]:
my_lasso_SGD.fit(X_train, y_train);

Epoch 0, Loss: 9468253.0012
Epoch 20, Loss: 1236995.0737
Epoch 40, Loss: 1133760.4365
Epoch 60, Loss: 1082465.8200
Epoch 80, Loss: 1054768.8939
Epoch 100, Loss: 1038755.4557
Epoch 120, Loss: 1028935.0248
Epoch 140, Loss: 1022603.6141
Epoch 160, Loss: 1018344.8501
Epoch 180, Loss: 1015387.2823


### - initialize and fit sklearn.linear_model.Lasso

In [130]:
lasso = sklearn.linear_model.Lasso()

In [131]:
lasso.fit(X_train, y_train)

,alpha,1.0
,fit_intercept,True
,precompute,False
,copy_X,True
,max_iter,1000
,tol,0.0001
,warm_start,False
,positive,False
,random_state,None
,selection,'cyclic'


### - calculate metrics and fill in the table

In [132]:
for table, metric in zip((results_r_squared, results_RMSE, results_MAE), (r_squared, sklearn.metrics.root_mean_squared_error, sklearn.metrics.mean_absolute_error)):
    for model, name, idx in zip((lasso, my_lasso_SGD), ('sklearn_default_lasso', 'my_lasso_SGD'), (6, 7)):
        table.loc[idx] = [
            name,
            metric(y_pred=model.predict(X_train), y_true=y_train),
            metric(y_pred=model.predict(X_test), y_true=y_test),
            metric(y_pred=model.predict(X_test_processed), y_true=y_test_processed)
        ]

In [133]:
results_r_squared[6:8]

,model,train,test,test_processed
6,sklearn_default_lasso,0.575,0.022,0.572
7,my_lasso_SGD,0.572,0.022,0.572


In [134]:
results_MAE[6:8]

,model,train,test,test_processed
6,sklearn_default_lasso,693.126,906.941,679.806
7,my_lasso_SGD,694.965,910.129,680.674


In [135]:
results_RMSE[6:8]

,model,train,test,test_processed
6,sklearn_default_lasso,1003.055,9606.533,948.340
7,my_lasso_SGD,1006.740,9605.322,948.273


## 3) my ElasticNet implementation 

In [136]:
from my_regression import MyElasticNet

### - initialize and fit MyElasticNet with SGD

In [137]:
my_elastic_net_SGD = MyElasticNet(method='SGD', l1_alpha=0.5, l2_alpha=0.6, eta=0.00001, batch_size=128, tol=200)

In [138]:
my_elastic_net_SGD.fit(X_train, y_train);

Epoch 0, Loss: 13971639.8713
Epoch 100, Loss: 1438984.1312
Epoch 200, Loss: 1381093.0179
Epoch 300, Loss: 1355388.0368


### - initialize and fit sklearn.linear_model.ElasticNet

In [139]:
elastic_net = sklearn.linear_model.ElasticNet()

In [140]:
elastic_net.fit(X_train, y_train)

,alpha,1.0
,l1_ratio,0.5
,fit_intercept,True
,precompute,False
,max_iter,1000
,copy_X,True
,tol,0.0001
,warm_start,False
,positive,False
,random_state,None
,selection,'cyclic'


### - calculate metrics and fill in the table

In [141]:
for table, metric in zip((results_r_squared, results_RMSE, results_MAE), (r_squared, sklearn.metrics.root_mean_squared_error, sklearn.metrics.mean_absolute_error)):
    for model, name, idx in zip((elastic_net, my_elastic_net_SGD), ('sklearn_default_elastic_net', 'my_elastic_net_SGD'), (8, 9)):
        table.loc[idx] = [
            name,
            metric(y_pred=model.predict(X_train), y_true=y_train),
            metric(y_pred=model.predict(X_test), y_true=y_test),
            metric(y_pred=model.predict(X_test_processed), y_true=y_test_processed)
        ]

In [142]:
results_r_squared[8:10]

,model,train,test,test_processed
8,sklearn_default_elastic_net,0.425,0.016,0.441
9,my_elastic_net_SGD,0.429,0.017,0.437


In [143]:
results_MAE[8:10]

,model,train,test,test_processed
8,sklearn_default_elastic_net,795.789,1029.829,776.615
9,my_elastic_net_SGD,792.938,1012.467,773.271


In [144]:
results_RMSE[8:10]

,model,train,test,test_processed
8,sklearn_default_elastic_net,1166.409,9633.879,1082.772
9,my_elastic_net_SGD,1163.005,9629.370,1087.408


# 6. Feature Normalization

For many machine learning models, it is important that quantitative data have the same scale. This applies to:

- algorithms that calculate **distance** (for example, the *k-nearest neighbors* algorithm or the *k-means method*)

- models that optimize weights using **gradient descent** and use **regularization** (in particular, linear or logistic **regression**)

Furthermore, in some cases, we may need to transform the data to make it follow a distribution closer to normal. This can be important for:

- Conducting **statistical tests**

- Transforming a nonlinear relationship into a linear one (which is important, in particular, for calculating linear correlation or using a linear model)

*Do not normalize when*:

- using tree-based algorithms

- interpretability of coefficients is needed

- the scale contains important information

## 1) MinMaxScaler

$$\Large x_i = \frac{x_i - x_{min}}{x_{max} - x_{min}}$$

In [145]:
from my_regression import MyMinMaxScaler

In [146]:
MyMinMaxScaler().fit_transform(X_train).values

array([[0.  , 0.25, 0.  , ..., 0.  , 0.  , 0.  ],
       [0.  , 0.5 , 1.  , ..., 0.  , 0.  , 0.  ],
       [0.  , 0.5 , 1.  , ..., 0.  , 0.  , 0.  ],
       ...,
       [0.  , 0.25, 1.  , ..., 0.  , 0.  , 0.  ],
       [0.  , 0.5 , 0.  , ..., 0.  , 0.  , 0.  ],
       [0.  , 0.75, 1.  , ..., 0.  , 0.  , 0.  ]], shape=(47745, 22))

In [147]:
sklearn.preprocessing.MinMaxScaler().fit_transform(X_train)

array([[0.  , 0.25, 0.  , ..., 0.  , 0.  , 0.  ],
       [0.  , 0.5 , 1.  , ..., 0.  , 0.  , 0.  ],
       [0.  , 0.5 , 1.  , ..., 0.  , 0.  , 0.  ],
       ...,
       [0.  , 0.25, 1.  , ..., 0.  , 0.  , 0.  ],
       [0.  , 0.5 , 0.  , ..., 0.  , 0.  , 0.  ],
       [0.  , 0.75, 1.  , ..., 0.  , 0.  , 0.  ]], shape=(47745, 22))

## 2) StandardScaler

$$\Large x_i = \frac{x_i - \mu}{\sigma}$$
$\large \mu$ - mean \
$\large \sigma$ - standard deviation

In [148]:
from my_regression import MyStandardScaler

In [149]:
MyStandardScaler().fit_transform(X_train).values

array([[-0.45533812, -0.48354232, -1.05272326, ..., -0.23787774,
        -0.23476542, -0.21640902],
       [-0.45533812,  0.45322895,  0.94991727, ..., -0.23787774,
        -0.23476542, -0.21640902],
       [-0.45533812,  0.45322895,  0.94991727, ..., -0.23787774,
        -0.23476542, -0.21640902],
       ...,
       [-0.45533812, -0.48354232,  0.94991727, ..., -0.23787774,
        -0.23476542, -0.21640902],
       [-0.45533812,  0.45322895, -1.05272326, ..., -0.23787774,
        -0.23476542, -0.21640902],
       [-0.45533812,  1.39000022,  0.94991727, ..., -0.23787774,
        -0.23476542, -0.21640902]], shape=(47745, 22))

In [150]:
sklearn.preprocessing.StandardScaler().fit_transform(X_train)

array([[-0.45533812, -0.48354232, -1.05272326, ..., -0.23787774,
        -0.23476542, -0.21640902],
       [-0.45533812,  0.45322895,  0.94991727, ..., -0.23787774,
        -0.23476542, -0.21640902],
       [-0.45533812,  0.45322895,  0.94991727, ..., -0.23787774,
        -0.23476542, -0.21640902],
       ...,
       [-0.45533812, -0.48354232,  0.94991727, ..., -0.23787774,
        -0.23476542, -0.21640902],
       [-0.45533812,  0.45322895, -1.05272326, ..., -0.23787774,
        -0.23476542, -0.21640902],
       [-0.45533812,  1.39000022,  0.94991727, ..., -0.23787774,
        -0.23476542, -0.21640902]], shape=(47745, 22))

# 7. Fit custom and sklearn models with normalized data

## 1) MinMaxScaler

In [151]:
minmax_lin_reg = sklearn.pipeline.make_pipeline(MyMinMaxScaler(), MyLinearRegressor(method='SGD', eta=0.001, tol=200, log_freq=5))
minmax_ridge = sklearn.pipeline.make_pipeline(MyMinMaxScaler(), MyRidge(method='SGD', alpha=0.001, eta=0.001, batch_size=256, tol=50, log_freq=100))
minmax_lasso = sklearn.pipeline.make_pipeline(MyMinMaxScaler(), MyLasso(method='SGD', alpha=0.01, eta=0.001, batch_size=256, tol=50, log_freq=100))
minmax_elastic_net = sklearn.pipeline.make_pipeline(MyMinMaxScaler(), MyElasticNet(method='SGD', l1_alpha=0.2, l2_alpha=0.3, eta=0.0001, batch_size=128, tol=100))

In [152]:
sk_minmax_lin_reg = sklearn.pipeline.make_pipeline(sklearn.preprocessing.MinMaxScaler(), sklearn.linear_model.LinearRegression())
sk_minmax_ridge = sklearn.pipeline.make_pipeline(sklearn.preprocessing.MinMaxScaler(), sklearn.linear_model.Ridge())
sk_minmax_lasso = sklearn.pipeline.make_pipeline(sklearn.preprocessing.MinMaxScaler(), sklearn.linear_model.Lasso())
sk_minmax_elastic_net = sklearn.pipeline.make_pipeline(sklearn.preprocessing.MinMaxScaler(), sklearn.linear_model.ElasticNet())

In [153]:
for ppl in (minmax_lin_reg, minmax_ridge, minmax_lasso, minmax_elastic_net, sk_minmax_lin_reg, sk_minmax_ridge, sk_minmax_lasso, sk_minmax_elastic_net):
    ppl.fit(X_train, y_train)

Epoch 0, Loss: 2765089.7830
Epoch 5, Loss: 1096470.0220
Epoch 10, Loss: 1034554.7275
Epoch 15, Loss: 1017944.1989
Epoch 20, Loss: 1011052.5458
Epoch 25, Loss: 1008058.4614
Epoch 30, Loss: 1006597.6449
Epoch 0, Loss: 7025167.3144
Epoch 100, Loss: 1027557.0039
Epoch 200, Loss: 1009791.9244
Epoch 0, Loss: 7024740.7578
Epoch 100, Loss: 1026070.0808
Epoch 200, Loss: 1008760.4847
Epoch 0, Loss: 12122330.7278
Epoch 100, Loss: 1923968.9314


In [154]:
for table, metric in zip((results_r_squared, results_RMSE, results_MAE), (r_squared, sklearn.metrics.root_mean_squared_error, sklearn.metrics.mean_absolute_error)):
    for ppl, name, idx in zip(
        (sk_minmax_lin_reg, minmax_lin_reg, sk_minmax_ridge, minmax_ridge, sk_minmax_lasso, minmax_lasso, sk_minmax_elastic_net, minmax_elastic_net),
        ('sk_minmax_lin_reg', 'my_minmax_lin_reg', 'sk_minmax_ridge', 'my_minmax_ridge', 'sk_minmax_lasso', 'my_minmax_lasso', 'sk_minmax_elastic_net', 'my_minmax_elastic_net'),
        (10, 11, 12, 13, 14, 15, 16, 17)):
        table.loc[idx] = [
            name,
            metric(y_pred=ppl.predict(X_train), y_true=y_train),
            metric(y_pred=ppl.predict(X_test), y_true=y_test),
            metric(y_pred=ppl.predict(X_test_processed), y_true=y_test_processed)
        ]

In [155]:
results_r_squared[10:18]

,model,train,test,test_processed
10,sk_minmax_lin_reg,0.575,0.022,0.571
11,my_minmax_lin_reg,0.575,0.022,0.572
12,sk_minmax_ridge,0.575,0.022,0.571
13,my_minmax_ridge,0.574,0.022,0.572
14,sk_minmax_lasso,0.575,0.022,0.572
15,my_minmax_lasso,0.574,0.022,0.572
16,sk_minmax_elastic_net,0.219,0.007,0.229
17,my_minmax_elastic_net,0.196,0.007,0.198


In [156]:
results_MAE[10:18]

,model,train,test,test_processed
10,sk_minmax_lin_reg,693.510,907.138,680.276
11,my_minmax_lin_reg,692.928,906.731,679.439
12,sk_minmax_ridge,693.506,907.140,680.265
13,my_minmax_ridge,693.955,907.976,680.031
14,sk_minmax_lasso,693.141,907.027,679.789
15,my_minmax_lasso,693.957,907.732,680.188
16,sk_minmax_elastic_net,965.059,1215.603,941.948
17,my_minmax_elastic_net,926.741,1167.893,906.806


In [157]:
results_RMSE[10:18]

,model,train,test,test_processed
10,sk_minmax_lin_reg,1002.852,9606.256,948.598
11,my_minmax_lin_reg,1003.200,9605.163,947.824
12,sk_minmax_ridge,1002.852,9606.245,948.576
13,my_minmax_ridge,1004.337,9604.065,947.480
14,sk_minmax_lasso,1003.063,9606.524,948.208
15,my_minmax_lasso,1003.989,9604.198,947.804
16,sk_minmax_elastic_net,1359.992,9676.800,1272.079
17,my_minmax_elastic_net,1379.436,9678.334,1297.150


## 2) StandardScaler

In [158]:
standard_lin_reg = sklearn.pipeline.make_pipeline(MyStandardScaler(), MyLinearRegressor(method='SGD', eta=0.001, tol=200, log_freq=5))
standard_ridge = sklearn.pipeline.make_pipeline(MyStandardScaler(), MyRidge(method='SGD', alpha=0.001, eta=0.001, batch_size=256, tol=50, log_freq=100))
standard_lasso = sklearn.pipeline.make_pipeline(MyStandardScaler(), MyLasso(method='SGD', alpha=0.01, eta=0.001, batch_size=256, tol=50, log_freq=100))
standard_elastic_net = sklearn.pipeline.make_pipeline(MyStandardScaler(), MyElasticNet(method='SGD', l1_alpha=0.1, l2_alpha=0.1, eta=0.0001, batch_size=128, tol=100))

In [159]:
sk_standard_lin_reg = sklearn.pipeline.make_pipeline(sklearn.preprocessing.StandardScaler(), sklearn.linear_model.LinearRegression())
sk_standard_ridge = sklearn.pipeline.make_pipeline(sklearn.preprocessing.StandardScaler(), sklearn.linear_model.Ridge())
sk_standard_lasso = sklearn.pipeline.make_pipeline(sklearn.preprocessing.StandardScaler(), sklearn.linear_model.Lasso())
sk_standard_elastic_net = sklearn.pipeline.make_pipeline(sklearn.preprocessing.StandardScaler(), sklearn.linear_model.ElasticNet())

In [160]:
for ppl in (standard_lin_reg, standard_ridge, standard_lasso, standard_elastic_net, sk_standard_lin_reg, sk_standard_ridge, sk_standard_lasso, sk_standard_elastic_net):
    ppl.fit(X_train, y_train)

Epoch 0, Loss: 3199684.1686
Epoch 5, Loss: 1003529.3848
Epoch 10, Loss: 1003427.4879
Epoch 0, Loss: 10400252.1924
Epoch 0, Loss: 10399855.0045
Epoch 0, Loss: 13605547.4672
Epoch 100, Loss: 1115188.3464


In [161]:
for table, metric in zip((results_r_squared, results_RMSE, results_MAE), (r_squared, sklearn.metrics.root_mean_squared_error, sklearn.metrics.mean_absolute_error)):
    for ppl, name, idx in zip(
        (sk_standard_lin_reg, standard_lin_reg, sk_standard_ridge, standard_ridge, sk_standard_lasso, standard_lasso, sk_standard_elastic_net, standard_elastic_net),
        ('sk_standard_lin_reg', 'my_standard_lin_reg', 'sk_standard_ridge', 'my_standard_ridge', 'sk_standard_lasso', 'my_standard_lasso', 'sk_standard_elastic_net', 'my_standard_elastic_net'),
        (18, 19, 20, 21, 22, 23, 24, 25)):
        table.loc[idx] = [
            name,
            metric(y_pred=ppl.predict(X_train), y_true=y_train),
            metric(y_pred=ppl.predict(X_test), y_true=y_test),
            metric(y_pred=ppl.predict(X_test_processed), y_true=y_test_processed)
        ]

In [162]:
results_r_squared[18:26]

,model,train,test,test_processed
18,sk_standard_lin_reg,0.575,0.022,0.571
19,my_standard_lin_reg,0.575,0.022,0.571
20,sk_standard_ridge,0.575,0.022,0.571
21,my_standard_ridge,0.575,0.022,0.571
22,sk_standard_lasso,0.575,0.022,0.571
23,my_standard_lasso,0.575,0.022,0.571
24,sk_standard_elastic_net,0.537,0.021,0.545
25,my_standard_elastic_net,0.529,0.019,0.531


In [163]:
results_MAE[18:26]

,model,train,test,test_processed
18,sk_standard_lin_reg,693.510,907.138,680.276
19,my_standard_lin_reg,693.178,906.619,679.781
20,sk_standard_ridge,693.510,907.138,680.275
21,my_standard_ridge,692.928,906.648,679.649
22,sk_standard_lasso,693.334,907.046,680.077
23,my_standard_lasso,693.398,907.049,680.138
24,sk_standard_elastic_net,722.029,945.242,705.876
25,my_standard_elastic_net,704.395,922.830,689.341


In [164]:
results_RMSE[18:26]

,model,train,test,test_processed
18,sk_standard_lin_reg,1002.852,9606.256,948.598
19,my_standard_lin_reg,1002.999,9606.251,948.430
20,sk_standard_ridge,1002.852,9606.256,948.597
21,my_standard_ridge,1002.961,9606.355,948.489
22,sk_standard_lasso,1002.882,9606.374,948.471
23,my_standard_lasso,1002.950,9606.291,948.609
24,sk_standard_elastic_net,1046.521,9612.033,976.777
25,my_standard_elastic_net,1055.752,9618.059,992.460


# 8. Overfit models

In [165]:
polynomial_lin_reg = sklearn.pipeline.make_pipeline(sklearn.preprocessing.PolynomialFeatures(degree=10), MyLinearRegressor(method='analytical', eta=1e-11, tol=200, batch_size=256, log_freq=100, max_iter=2000))
polynomial_ridge = sklearn.pipeline.make_pipeline(sklearn.preprocessing.PolynomialFeatures(degree=10), MyRidge(method='analytical', alpha=0, batch_size=256, tol=1e-10, log_freq=100, max_iter=5000))
polynomial_lasso = sklearn.pipeline.make_pipeline(sklearn.preprocessing.PolynomialFeatures(degree=10), sklearn.preprocessing.StandardScaler(), MyLasso(method='SGD', alpha=100, eta=0.001, batch_size=128, tol=500, log_freq=10))
polynomial_elastic_net = sklearn.pipeline.make_pipeline(sklearn.preprocessing.PolynomialFeatures(degree=10), sklearn.preprocessing.StandardScaler(), MyElasticNet(method='SGD', l1_alpha=0.1, l2_alpha=0.1, eta=0.001, batch_size=128, tol=500, log_freq=10))

In [166]:
sk_polynomial_lin_reg = sklearn.pipeline.make_pipeline(sklearn.preprocessing.PolynomialFeatures(degree=10), sklearn.linear_model.LinearRegression())
sk_polynomial_ridge = sklearn.pipeline.make_pipeline(sklearn.preprocessing.PolynomialFeatures(degree=10), sklearn.linear_model.Ridge(alpha=0))
sk_polynomial_lasso = sklearn.pipeline.make_pipeline(sklearn.preprocessing.PolynomialFeatures(degree=10), sklearn.linear_model.Lasso())
sk_polynomial_elastic_net = sklearn.pipeline.make_pipeline(sklearn.preprocessing.PolynomialFeatures(degree=10), sklearn.linear_model.ElasticNet())

In [167]:
sk_polynomial_ridge

,steps,"[('polynomialfeatures', ...), ('ridge', ...)]"
,transform_input,None
,memory,None
,verbose,False
,degree,10
,interaction_only,False
,include_bias,True
,order,'C'
,alpha,0
,fit_intercept,True
,copy_X,True


In [168]:
for ppl in (polynomial_lin_reg, polynomial_ridge, polynomial_lasso, polynomial_elastic_net, sk_polynomial_lin_reg, sk_polynomial_ridge, sk_polynomial_lasso, sk_polynomial_elastic_net):
    ppl.fit(X_train[['bathrooms', 'bedrooms']], y_train)

Epoch 0, Loss: 7741588.8048
Epoch 10, Loss: 1178520.5845
Epoch 20, Loss: 1176378.6691
Epoch 30, Loss: 1174246.4000
Epoch 40, Loss: 1173655.1936
Epoch 50, Loss: 1171750.3353
Epoch 60, Loss: 1171672.2174
Epoch 70, Loss: 1171326.2729
Epoch 80, Loss: 1172733.7024
Epoch 90, Loss: 1172569.8866
Epoch 100, Loss: 1172670.0817
Epoch 110, Loss: 1172125.2363
Epoch 0, Loss: 7747726.0932
Epoch 10, Loss: 1265253.8601
Epoch 20, Loss: 1262484.5064
Epoch 30, Loss: 1261797.8662
Epoch 40, Loss: 1260554.8058
Epoch 50, Loss: 1259489.6843


In [169]:
for table, metric in zip((results_r_squared, results_RMSE, results_MAE), (r_squared, sklearn.metrics.root_mean_squared_error, sklearn.metrics.mean_absolute_error)):
    for ppl, name, idx in zip(
        (sk_polynomial_lin_reg, polynomial_lin_reg, sk_polynomial_ridge, polynomial_ridge, sk_polynomial_lasso, polynomial_lasso, sk_polynomial_elastic_net, polynomial_elastic_net),
        ('sk_polynomial_lin_reg', 'my_polynomial_lin_reg', 'sk_polynomial_ridge', 'my_polynomial_ridge', 'sk_polynomial_lasso', 'my_polynomial_lasso', 'sk_polynomial_elastic_net', 'my_polynomial_elastic_net'),
        (26, 27, 28, 29, 30, 31, 32, 33)):
        table.loc[idx] = [
            name,
            metric(y_pred=ppl.predict(X_train[['bathrooms', 'bedrooms']]), y_true=y_train),
            metric(y_pred=ppl.predict(X_test[['bathrooms', 'bedrooms']]), y_true=y_test),
            metric(y_pred=ppl.predict(X_test_processed[['bathrooms', 'bedrooms']]), y_true=y_test_processed)
        ]

In [170]:
results_r_squared[26:34]

,model,train,test,test_processed
26,sk_polynomial_lin_reg,0.522,-11047693584507629161768185495552.000,0.495
27,my_polynomial_lin_reg,0.522,-5448168614625911528904196096.000,0.423
28,sk_polynomial_ridge,0.520,-1142637537302703389705334907666272694305019658...,-29552529814201484.000
29,my_polynomial_ridge,0.521,-3072150947091243636526787244474258957477819187...,-50649827803039600.000
30,sk_polynomial_lasso,0.519,-110602438118956158322475008.000,0.499
31,my_polynomial_lasso,0.501,-1350331356589088000.000,0.492
32,sk_polynomial_elastic_net,0.514,-181337715193201366797385728.000,0.494
33,my_polynomial_elastic_net,0.463,-1725005507572610036137984.000,0.453


In [171]:
results_MAE[26:34]

,model,train,test,test_processed
26,sk_polynomial_lin_reg,748.960,118154243413034576.000,737.965
27,my_polynomial_lin_reg,748.960,2623848740781653.000,740.220
28,sk_polynomial_ridge,749.740,12016216991224005918720000.000,2074430700.363
29,my_polynomial_ridge,749.345,6230670399232710138658816.000,2715754757.500
30,sk_polynomial_lasso,750.387,373848697528033.438,738.547
31,my_polynomial_lasso,759.823,41307957584.753,745.501
32,sk_polynomial_elastic_net,756.750,478693567023206.688,745.395
33,my_polynomial_elastic_net,776.443,46688375894609.094,761.246


In [172]:
results_RMSE[26:34]

,model,train,test,test_processed
26,sk_polynomial_lin_reg,1064.247,32284226572736806912.000,1029.596
27,my_polynomial_lin_reg,1064.247,716935105215709568.000,1100.620
28,sk_polynomial_ridge,1065.809,3283286795007563681193000960.000,249066396937.060
29,my_polynomial_ridge,1064.597,1702455824729596274210242560.000,326066958682.910
30,sk_polynomial_lasso,1067.468,102149660305800384.000,1025.589
31,my_polynomial_lasso,1086.518,11286901961638.285,1032.604
32,sk_polynomial_elastic_net,1072.339,130797259953387056.000,1030.832
33,my_polynomial_elastic_net,1127.032,12757037125906160.000,1071.491


# 9. Native models

## 1) mean

In [173]:
for table, metric in zip((results_r_squared, results_RMSE, results_MAE), (r_squared, sklearn.metrics.root_mean_squared_error, sklearn.metrics.mean_absolute_error)):
        table.loc[34] = [
            'native_mean',
            metric(y_pred=np.full(y_train.shape[0], fill_value=y_train.mean()), y_true=y_train),
            metric(y_pred=np.full(y_test.shape[0], fill_value=y_test.mean()), y_true=y_test),
            metric(y_pred=np.full(y_test_processed.shape[0], fill_value=y_test_processed.mean()), y_true=y_test_processed)
        ]

## 2) median

In [174]:
for table, metric in zip((results_r_squared, results_RMSE, results_MAE), (r_squared, sklearn.metrics.root_mean_squared_error, sklearn.metrics.mean_absolute_error)):
        table.loc[35] = [
            'native_median',
            metric(y_pred=np.full(y_train.shape[0], fill_value=y_train.median()), y_true=y_train),
            metric(y_pred=np.full(y_test.shape[0], fill_value=y_test.median()), y_true=y_test),
            metric(y_pred=np.full(y_test_processed.shape[0], fill_value=y_test_processed.median()), y_true=y_test_processed)
        ]

In [175]:
results_RMSE

,model,train,test,test_processed
0,sklearn_default_lin_reg,1002.852,9606.256,948.598
1,my_lin_reg_analytical,1002.852,9606.256,948.598
2,my_lin_reg_SGD,1003.378,9606.887,947.880
3,sklearn_default_ridge,1002.852,9606.253,948.591
4,my_ridge_analytical,1002.852,9606.255,948.593
5,my_ridge_SGD,1004.626,9605.439,947.521
6,sklearn_default_lasso,1003.055,9606.533,948.340
7,my_lasso_SGD,1006.740,9605.322,948.273
8,sklearn_default_elastic_net,1166.409,9633.879,1082.772
9,my_elastic_net_SGD,1163.005,9629.370,1087.408


In [176]:
results_MAE

,model,train,test,test_processed
0,sklearn_default_lin_reg,693.510,907.138,680.276
1,my_lin_reg_analytical,693.510,907.138,680.276
2,my_lin_reg_SGD,690.221,904.594,676.841
3,sklearn_default_ridge,693.507,907.136,680.270
4,my_ridge_analytical,693.506,907.134,680.270
5,my_ridge_SGD,693.039,907.594,679.125
6,sklearn_default_lasso,693.126,906.941,679.806
7,my_lasso_SGD,694.965,910.129,680.674
8,sklearn_default_elastic_net,795.789,1029.829,776.615
9,my_elastic_net_SGD,792.938,1012.467,773.271


In [177]:
results_r_squared

,model,train,test,test_processed
0,sklearn_default_lin_reg,0.575,0.022,0.571
1,my_lin_reg_analytical,0.575,0.022,0.571
2,my_lin_reg_SGD,0.575,0.022,0.572
3,sklearn_default_ridge,0.575,0.022,0.571
4,my_ridge_analytical,0.575,0.022,0.571
5,my_ridge_SGD,0.574,0.022,0.572
6,sklearn_default_lasso,0.575,0.022,0.572
7,my_lasso_SGD,0.572,0.022,0.572
8,sklearn_default_elastic_net,0.425,0.016,0.441
9,my_elastic_net_SGD,0.429,0.017,0.437


In [178]:
top_7_by_RMSE = set(results_RMSE[results_RMSE['model'].apply(lambda x: not x.startswith('sk'))].sort_values(by='test_processed').head(7).model)
top_7_by_MAE = set(results_MAE[results_MAE['model'].apply(lambda x: not x.startswith('sk'))].sort_values(by='test_processed').head(7).model)
top_7_by_R2 = set(results_r_squared[results_r_squared['model'].apply(lambda x: not x.startswith('sk'))].sort_values(by='test_processed', ascending=False).head(7).model)

In [179]:
top_7_by_RMSE & top_7_by_MAE & top_7_by_R2

{'my_lin_reg_SGD',
 'my_minmax_lin_reg',
 'my_minmax_ridge',
 'my_ridge_SGD',
 'my_standard_lin_reg'}

Best models are:
- Linear REgression (SDG)
- MinMaxScaler + Linear Regression
- MinMaxScaler + Ridge
- Ridge (SGD)
- StandardScaler + Linear Regression